# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# sns.set_theme(style="whitegrid")
# plt.rcParams['figure.figsize'] = (10, 6)

file_path = 'База_док_21_07_2026_Коррекция_по_нозологии_финал_.csv'
df = pd.read_csv(file_path, sep=';', encoding='utf-8', low_memory=False)

df.columns = df.columns.str.strip()

target_survival = 'Исход 1- жив,0 -мерт'
target_amputation = 'Сохранение конечности 1+,2-'

df[target_survival] = pd.to_numeric(df[target_survival], errors='coerce')
df[target_amputation] = pd.to_numeric(df[target_amputation], errors='coerce')

num_features = ['Глюкоза', 'креатинин', 'холестерин', 'СРБ', 'wbc', 'Hb']
cat_features = ['диабет', 'ИБС', 'Инфаркт анамнез (да/нет)', 'ОНМК анамнез (да/нет)']

# ИСПРАВЛЕННЫЙ БЛОК ОЧИСТКИ ЧИСЛОВЫХ ПРИЗНАКОВ
for col in num_features:
    if col in df.columns:
        temp_col = df[col].astype(str).str.replace(',', '.', regex=False)
        # Извлекаем только числа (игнорируя текст типа "мкмоль/л")
        temp_col = temp_col.str.extract(r'(\d+\.?\d*)')[0]
        df[col] = pd.to_numeric(temp_col, errors='coerce')

In [ ]:
print(df[target_survival].value_counts(dropna=False))

In [ ]:
print("="*60)
print("ЧАСТЬ 1: АНАЛИЗ ПО ИСХОДУ (ВЫЖИВАЕМОСТЬ)")
print("="*60)
print("Распределение классов (1.0 - жив, 0.0 - мертв, NaN - пропуск):")
print(df[target_survival].value_counts(dropna=False))
print("\n")

print("--- Средние и медианные значения числовых показателей в зависимости от исхода ---")
numeric_stats_survival = df.groupby(target_survival)[num_features].agg(['mean', 'median']).T
print(numeric_stats_survival.round(2))
print("\n")

print("--- Распределение сопутствующих заболеваний в зависимости от исхода (кол-во пациентов) ---")
for cat in cat_features:
    if cat in df.columns:
        print(f"\nТаблица сопряженности для: {cat}")
        print(pd.crosstab(df[cat], df[target_survival], dropna=False, margins=True))


print("\n\n" + "="*60)
print("ЧАСТЬ 2: АНАЛИЗ ПО СОХРАНЕНИЮ КОНЕЧНОСТИ (АМПУТАЦИЯ)")
print("="*60)
print("Распределение классов (1.0 - сохранена, 2.0 - ампутация, NaN - пропуск):")
print(df[target_amputation].value_counts(dropna=False))
print("\n")

print("--- Средние и медианные значения числовых показателей в зависимости от ампутации ---")
numeric_stats_amp = df.groupby(target_amputation)[num_features].agg(['mean', 'median']).T
print(numeric_stats_amp.round(2))
print("\n")

print("--- Распределение сопутствующих заболеваний в зависимости от ампутации (кол-во пациентов) ---")
for cat in cat_features:
    if cat in df.columns:
        print(f"\nТаблица сопряженности для: {cat}")
        print(pd.crosstab(df[cat], df[target_amputation], dropna=False, margins=True))